# WTI Review Runner

Run the company review sequence for both daily and weekly WTI experiments.

Default batch order:

1. Combined daily+weekly report setting requested for the company update
2. Combined daily+weekly MSE + robust stable improvement run

If needed, you can add the raw reference or validation-size sweep by uncommenting them in `BATCH_CONFIG_RELATIVE_PATHS`.


In [ ]:
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

BOOTSTRAP_SENTINEL = Path("/content/newoil_colab_bootstrap_v3")

PACKAGE_SPECS = [
    "numpy>=1.26,<2.2",
    "pandas==2.2.2",
    "protobuf>=4.25,<6",
    "tensorboard>=2.18,<2.20",
    "pyyaml==6.0.2",
    "openpyxl==3.1.5",
    "utilsforecast",
    "coreforecast",
    "lightning-utilities>=0.11,<0.16",
    "torchmetrics>=1.6,<1.9",
    "pytorch-lightning>=2.4,<2.6",
    "ray[tune]>=2.20,<3.0",
]
NEURALFORECAST_SPEC = "neuralforecast==3.1.7"

OPTIONAL_TORCH_PACKAGES_TO_REMOVE = [
    "torchvision",
    "torchaudio",
]

CRITICAL_IMPORTS = [
    "numpy",
    "pandas",
    "yaml",
    "openpyxl",
    "utilsforecast",
    "coreforecast",
    "torch",
    "pytorch_lightning",
    "torchmetrics",
    "ray",
    "neuralforecast",
]


def import_is_healthy(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except Exception as exc:
        print(f"Import check failed for {module_name}: {type(exc).__name__}: {exc}")
        return False

installed_optional_torch_packages = [
    package
    for package in OPTIONAL_TORCH_PACKAGES_TO_REMOVE
    if importlib.util.find_spec(package) is not None
]

bootstrap_changed = False

if installed_optional_torch_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", *installed_optional_torch_packages],
        check=True,
    )
    bootstrap_changed = True

needs_repair = (not BOOTSTRAP_SENTINEL.exists()) or any(
    not import_is_healthy(module_name) for module_name in CRITICAL_IMPORTS
)

if needs_repair:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade", *PACKAGE_SPECS],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-cache-dir",
            "--no-deps",
            "--upgrade",
            NEURALFORECAST_SPEC,
        ],
        check=True,
    )
    bootstrap_changed = True

if bootstrap_changed:
    BOOTSTRAP_SENTINEL.write_text("ready\n", encoding="utf-8")
    print("Bootstrap installed/repaired runtime packages. Restarting the kernel once for a clean import state.")
    os.kill(os.getpid(), 9)

sanity_code = "import numpy, pandas, torch, pytorch_lightning, torchmetrics, ray, neuralforecast; print('import sanity ok')"
sanity = subprocess.run([sys.executable, "-c", sanity_code], capture_output=True, text=True)
if sanity.returncode != 0:
    print(sanity.stdout)
    print(sanity.stderr)
    raise RuntimeError("Colab package sanity check failed before importing newoil. Restart runtime and rerun this cell.")

import pandas as pd

REPO_URL = "https://github.com/Jaeho777/newoil.git"
WORKDIR = Path("/content/newoil")

# If you received an updated weekly CSV, put it in Google Drive and set the full path below.
# Example: Path("/content/drive/MyDrive/newoil_inputs/0428DB_weekly.csv")
UPDATED_WEEKLY_DATA_SOURCE_PATH = None

BATCH_CONFIG_RELATIVE_PATHS = [
    "configs/batches/company_wti_mse_report.yaml",
    "configs/batches/company_wti_mse_scaled_regularized.yaml",
    # "configs/batches/weekly_wti_h2_mse_raw_report.yaml",
    # "configs/batches/weekly_wti_h2_mse_scaled_val_sweep.yaml",
]

SAVE_TO_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/newoil_outputs")
LOCAL_OUTPUT_ROOT = Path("/content/newoil_outputs")

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)], check=True)
commit_hash = subprocess.check_output(
    ["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print(f"Using newoil commit: {commit_hash}")
src_path = str(WORKDIR / "src")
sys.path = [path for path in sys.path if path != src_path]
sys.path.insert(0, src_path)

for module_name in list(sys.modules):
    if module_name == "newoil" or module_name.startswith("newoil."):
        del sys.modules[module_name]

from newoil import build_company_master_report, run_batch_from_config

repo_root = WORKDIR
output_root = DRIVE_OUTPUT_ROOT if SAVE_TO_GOOGLE_DRIVE else LOCAL_OUTPUT_ROOT
output_root.mkdir(parents=True, exist_ok=True)

if UPDATED_WEEKLY_DATA_SOURCE_PATH:
    source_path = Path(UPDATED_WEEKLY_DATA_SOURCE_PATH)
    target_path = repo_root / "data" / "0428DB_weekly.csv"
    print(f"Replacing weekly data: {source_path} -> {target_path}")
    shutil.copy2(source_path, target_path)
else:
    print("Using weekly data committed in the repository.")

all_summaries = []
batch_dirs = []
batch_results = []
for relative_path in BATCH_CONFIG_RELATIVE_PATHS:
    batch_config_path = repo_root / relative_path
    print(f"\n[RUN BATCH] {batch_config_path}")
    result = run_batch_from_config(
        batch_config_path=batch_config_path,
        repo_root=repo_root,
        output_root=output_root,
    )
    batch_results.append(result)
    summary_df = result.summary_df.copy()
    summary_df["batch_name"] = Path(relative_path).name
    all_summaries.append(summary_df)
    batch_dirs.append(str(result.batch_dir))

combined_summary_df = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
combined_summary_path = output_root / "wti_review_combined_summary.csv"
combined_summary_df.to_csv(combined_summary_path, index=False)
master_report_path = build_company_master_report(batch_results, output_root)

print("\nBatch directories:")
for batch_dir in batch_dirs:
    print(batch_dir)

print(f"\nCombined summary: {combined_summary_path}")
print(f"Master report: {master_report_path}")
combined_summary_df
